<a href="https://colab.research.google.com/github/deva123932/CSA-4703-DL/blob/main/exp_18.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

```markdown
# Experiment 18: RNN, LSTM, and GRU Hyperparameter Experimentation

## Step 1: Data Preparation
We will import the libraries, load the IMDB movie reviews dataset, tokenize, and pad the sequences to set up our embedding layer.
```

In [1]:
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np
import matplotlib.pyplot as plt
import time

# Install keras-tuner if not already installed
!pip install -q -U keras-tuner
import keras_tuner as kt

# Load dataset
max_features = 10000
max_len = 150

(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=max_features)

# Pad sequences to ensure uniform length
x_train = pad_sequences(x_train, maxlen=max_len)
x_test = pad_sequences(x_test, maxlen=max_len)

print(f"x_train shape: {x_train.shape}")
print(f"x_test shape: {x_test.shape}")

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
x_train shape: (25000, 150)
x_test shape: (25000, 150)


```markdown
## Step 2: Model-Builder Function
We will define a flexible model builder function that allows KerasTuner to select the cell type (SimpleRNN, LSTM, or GRU), the number of units, layers, dropout rates, learning rate, and embedding dimensions.
```

In [2]:
def build_model(hp):
    model = tf.keras.Sequential()

    # Tune embedding output dimension
    embedding_dim = hp.Int('embedding_dim', min_value=32, max_value=128, step=32)
    model.add(tf.keras.layers.Embedding(input_dim=max_features, output_dim=embedding_dim, input_length=max_len))

    # Tune recurrent cell type
    cell_type = hp.Choice('cell_type', values=['RNN', 'LSTM', 'GRU'])

    # Tune number of layers
    num_layers = hp.Int('num_layers', min_value=1, max_value=2)

    # Tune dropout
    dropout_rate = hp.Float('dropout', min_value=0.0, max_value=0.5, step=0.1)

    for i in range(num_layers):
        units = hp.Int(f'units_{i}', min_value=32, max_value=128, step=32)
        return_sequences = (i < num_layers - 1)

        if cell_type == 'RNN':
            model.add(tf.keras.layers.SimpleRNN(units=units, return_sequences=return_sequences, dropout=dropout_rate))
        elif cell_type == 'LSTM':
            model.add(tf.keras.layers.LSTM(units=units, return_sequences=return_sequences, dropout=dropout_rate))
        elif cell_type == 'GRU':
            model.add(tf.keras.layers.GRU(units=units, return_sequences=return_sequences, dropout=dropout_rate))

    model.add(tf.keras.layers.Dense(1, activation='sigmoid'))

    # Tune learning rate
    learning_rate = hp.Choice('learning_rate', values=[1e-2, 1e-3, 1e-4])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

```markdown
## Step 3: Run Hyperparameter Search
We will use `RandomSearch` from KerasTuner to search for the best model architecture and settings under a constrained budget (e.g., 5 trials, 3 epochs each) to save runtime while exploring multiple structures.
```

In [3]:
tuner = kt.RandomSearch(
    build_model,
    objective='val_accuracy',
    max_trials=5,
    executions_per_trial=1,
    directory='tuner_results',
    project_name='rnn_lstm_gru_tuning'
)

# Run the hyperparameter search
tuner.search(
    x_train, y_train,
    epochs=3,
    validation_split=0.2,
    batch_size=128
)

Trial 5 Complete [00h 00m 49s]
val_accuracy: 0.7472000122070312

Best val_accuracy So Far: 0.8737999796867371
Total elapsed time: 00h 13m 34s


```markdown
## Step 4: Compare Architectures and Results
We will query the best configurations, train the best configuration to completion, and compile the final performance comparison.
```

In [4]:
# Retrieve and display the best trial results
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]

print("--- Best Architecture Configuration Found ---")
print(f"Cell Type: {best_hps.get('cell_type')}")
print(f"Embedding Dim: {best_hps.get('embedding_dim')}")
print(f"Number of Layers: {best_hps.get('num_layers')}")
print(f"Dropout: {best_hps.get('dropout')}")
print(f"Learning Rate: {best_hps.get('learning_rate')}")

# Train best model
best_model = tuner.hypermodel.build(best_hps)

start_time = time.time()
history = best_model.fit(x_train, y_train, epochs=5, batch_size=128, validation_data=(x_test, y_test))
training_time = time.time() - start_time

loss, accuracy = best_model.evaluate(x_test, y_test)

print(f"\nTest Accuracy: {accuracy:.4f}")
print(f"Parameters Count: {best_model.count_params():,}")
print(f"Training Time (5 epochs): {training_time:.2f} seconds")

--- Best Architecture Configuration Found ---
Cell Type: GRU
Embedding Dim: 96
Number of Layers: 2
Dropout: 0.1
Learning Rate: 0.01
Epoch 1/5
196/196 ━━━━━━━━━━━━━━━━━━━━ 155s 771ms/step - accuracy: 0.7789 - loss: 0.4467 - val_accuracy: 0.8796 - val_loss: 0.2883
Epoch 2/5
196/196 ━━━━━━━━━━━━━━━━━━━━ 151s 767ms/step - accuracy: 0.9155 - loss: 0.2153 - val_accuracy: 0.8779 - val_loss: 0.2948
Epoch 3/5
196/196 ━━━━━━━━━━━━━━━━━━━━ 211s 818ms/step - accuracy: 0.9498 - loss: 0.1406 - val_accuracy: 0.8682 - val_loss: 0.3539
Epoch 4/5
196/196 ━━━━━━━━━━━━━━━━━━━━ 151s 773ms/step - accuracy: 0.9650 - loss: 0.1014 - val_accuracy: 0.8597 - val_loss: 0.4222
Epoch 5/5
196/196 ━━━━━━━━━━━━━━━━━━━━ 150s 766ms/step - accuracy: 0.9646 - loss: 0.0968 - val_accuracy: 0.8383 - val_loss: 0.4458
782/782 ━━━━━━━━━━━━━━━━━━━━ 38s 49ms/step - accuracy: 0.8383 - loss: 0.4458

Test Accuracy: 0.8383
Parameters Count: 1,062,369
Training Time (5 epochs): 818.10 seconds
